# Step 1 — Data Loading via TableShift

**Thesis:** Drift-Aware Selective Updating of Two-Stage Tabular ML Pipelines  
**Goal:** Load the `adult` (Income) and `diabetes_readmission` datasets through TableShift, split each into reference / pre-drift / post-drift sets, and inspect shapes, class balance, and feature types.

Reference implementation: `drift_framework/data/loader.py`


## 1.1 Environment setup


In [2]:
# Verify Python version (must be 3.10+)
import sys

print(f"Python {sys.version}")
assert sys.version_info >= (3, 10), "Python 3.10+ required"

Python 3.10.20 (main, May 10 2026, 19:31:32) [MSC v.1944 64 bit (AMD64)]


In [3]:
import sys, os

# Make sure drift_framework is importable from project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\jeffr\GitHub\pfc1


## 1.2 Imports and configuration


In [4]:
from dataclasses import dataclass
from typing import Dict, Tuple

import pandas as pd

# tableshift
from tableshift import get_dataset
from tableshift.core.features import PreprocessorConfig

SEED = 42
CACHE_DIR = "tableshift_cache"  # tableshift downloads data here

print("Imports OK")

Imports OK


## 1.3 Helper: passthrough preprocessor

TableShift applies its own preprocessing by default. We skip it so our custom two-stage pipeline can apply QuantileTransformer / TargetEncoder / StandardScaler on the raw values.


In [5]:
def passthrough_preprocessor() -> PreprocessorConfig:
    """Return a PreprocessorConfig that skips all TableShift transformations."""
    return PreprocessorConfig(
        categorical_features="passthrough",
        numeric_features="passthrough",
        dropna="all",
    )

## 1.4 DataBundle — container for a dataset's three splits


In [6]:
@dataclass
class DataBundle:
    """Holds reference, pre-drift, and post-drift splits plus feature metadata."""

    name: str
    X_ref: pd.DataFrame
    y_ref: pd.Series
    X_pre: pd.DataFrame
    y_pre: pd.Series
    X_post: pd.DataFrame
    y_post: pd.Series
    num_features: list
    cat_features: list

## 1.5 Feature-type detection


In [7]:
def detect_feature_types(X: pd.DataFrame) -> Tuple[list, list]:
    """Return (numeric_cols, categorical_cols) inferred from column dtypes."""
    cat_cols = [
        c
        for c in X.columns
        if isinstance(X[c].dtype, pd.CategoricalDtype)
        or pd.api.types.is_object_dtype(X[c])
    ]
    num_cols = [c for c in X.columns if c not in cat_cols]
    return num_cols, cat_cols

## 1.6 Split map

Each dataset uses a different splitting strategy in TableShift:

- `adult` → FixedSplitter → `train / validation / test`
- `diabetes_readmission` → DomainSplitter → `train / id_test / ood_test`

We map thesis roles (reference / pre-drift / post-drift) to the correct TableShift split names.


In [8]:
SPLIT_MAP: Dict[str, Dict[str, str]] = {
    "adult": {
        "reference": "train",
        "pre_drift": "validation",
        "post_drift": "test",
    },
    "diabetes_readmission": {
        "reference": "train",
        "pre_drift": "id_test",
        "post_drift": "ood_test",
    },
}

## 1.7 Dataset loader function


In [9]:
def load_dataset(name: str, cache_dir: str = CACHE_DIR) -> DataBundle:
    """Load a TableShift dataset and return reference, pre-drift, and post-drift splits."""
    if name not in SPLIT_MAP:
        raise ValueError(
            f"Dataset '{name}' not supported. Choose from: {list(SPLIT_MAP.keys())}"
        )

    print(f"\n{'='*60}")
    print(f"Loading dataset: {name}")
    print(f"{'='*60}")

    dset = get_dataset(
        name,
        cache_dir=cache_dir,
        preprocessor_config=passthrough_preprocessor(),
    )

    bundles = {}
    for role, split_name in SPLIT_MAP[name].items():
        X, y, _groups, _domain = dset.get_pandas(split_name)
        bundles[role] = (X.reset_index(drop=True), y.reset_index(drop=True))

    X_ref, y_ref = bundles["reference"]
    X_pre, y_pre = bundles["pre_drift"]
    X_post, y_post = bundles["post_drift"]

    num_features, cat_features = detect_feature_types(X_ref)

    return DataBundle(
        name=name,
        X_ref=X_ref,
        y_ref=y_ref,
        X_pre=X_pre,
        y_pre=y_pre,
        X_post=X_post,
        y_post=y_post,
        num_features=num_features,
        cat_features=cat_features,
    )

## 1.8 Load and inspect the `adult` dataset


In [10]:
adult = load_dataset("adult")


Loading dataset: adult


c:\Users\jeffr\GitHub\pfc1\.venv\lib\site-packages\tableshift\datasets\adult.py:99: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Target'] = df['Target'].replace(
c:\Users\jeffr\GitHub\pfc1\.venv\lib\site-packages\sklearn\compose\_column_transformer.py:1590: UserWarning: Instantiating CategoricalDtype without any arguments.Pass a CategoricalDtype instance to silence this warning.
  df_row = df_row.select_dtypes(
c:\Users\jeffr\GitHub\pfc1\.venv\lib\site-packages\sklearn\compose\_column_transformer.py:1590: UserWarning: Instantiating CategoricalDtype without any arguments.Pass a CategoricalDtype instance to silence this warning.
  df_row = df_row.select_dtypes(


In [11]:
def inspect_bundle(b: DataBundle):
    """Print shape, class balance, and feature info for all three splits."""
    for split_name, (X, y) in [
        ("Reference", (b.X_ref, b.y_ref)),
        ("Pre-drift", (b.X_pre, b.y_pre)),
        ("Post-drift", (b.X_post, b.y_post)),
    ]:
        print(f"{split_name:12s} | shape={X.shape} | positive_rate={y.mean():.3f}")

    print(f"\nNumeric features  ({len(b.num_features)}): {b.num_features}")
    print(f"Categorical features ({len(b.cat_features)}): {b.cat_features}")


print(f"\n--- Dataset: {adult.name} ---")
inspect_bundle(adult)


--- Dataset: adult ---
Reference    | shape=(24420, 12) | positive_rate=0.240
Pre-drift    | shape=(8141, 12) | positive_rate=0.244
Post-drift   | shape=(16281, 12) | positive_rate=0.236

Numeric features  (6): ['Age', 'Race', 'Sex', 'Capital Gain', 'Capital Loss', 'Hours per week']
Categorical features (6): ['Workclass', 'Education-Num', 'Marital Status', 'Occupation', 'Relationship', 'Country']


In [12]:
# Preview reference split
adult.X_ref.head(3)

,Age,Workclass,Education-Num,Marital Status,Occupation,Relationship,Race,Sex,Capital Gain,Capital Loss,Hours per week,Country
0,28.0,Self-emp-inc,10.0,Married-civ-spouse,Exec-managerial,Husband,1,1,0.0,0.0,70.0,United-States
1,21.0,nan,10.0,Never-married,nan,Unmarried,0,0,0.0,0.0,40.0,United-States
2,33.0,Local-gov,11.0,Married-civ-spouse,Prof-specialty,Husband,1,1,0.0,0.0,60.0,United-States


In [13]:
# Label distribution in reference split
print("Label distribution (reference):")
print(adult.y_ref.value_counts())

Label distribution (reference):
Target
0    18566
1     5854
Name: count, dtype: int64


## 1.9 Load and inspect the `diabetes_readmission` dataset


In [14]:
diabetes = load_dataset("diabetes_readmission")

print(f"\n--- Dataset: {diabetes.name} ---")
inspect_bundle(diabetes)


Loading dataset: diabetes_readmission


c:\Users\jeffr\GitHub\pfc1\.venv\lib\site-packages\sklearn\compose\_column_transformer.py:1590: UserWarning: Instantiating CategoricalDtype without any arguments.Pass a CategoricalDtype instance to silence this warning.
  df_row = df_row.select_dtypes(
c:\Users\jeffr\GitHub\pfc1\.venv\lib\site-packages\sklearn\compose\_column_transformer.py:1590: UserWarning: Instantiating CategoricalDtype without any arguments.Pass a CategoricalDtype instance to silence this warning.
  df_row = df_row.select_dtypes(



--- Dataset: diabetes_readmission ---
Reference    | shape=(34288, 46) | positive_rate=0.424
Pre-drift    | shape=(4287, 46) | positive_rate=0.416
Post-drift   | shape=(50968, 46) | positive_rate=0.494

Numeric features  (12): ['race', 'gender', 'admission_type_id', 'discharge_disposition_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']
Categorical features (34): ['age', 'weight', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetes

In [15]:
diabetes.X_ref.head(3)

,race,gender,age,weight,admission_type_id,discharge_disposition_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,...,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed
0,1,1,[70-80),nan,3,1,7.0,nan,nan,28.0,...,No,No,No,No,No,No,No,No,Ch,Yes
1,1,1,[80-90),nan,3,1,9.0,nan,nan,48.0,...,No,No,No,No,No,No,No,No,No,Yes
2,1,1,[50-60),nan,2,22,8.0,nan,InternalMedicine,41.0,...,No,No,Steady,No,No,No,No,No,Ch,Yes


## 1.10 Sanity checks


In [16]:
for bundle in [adult, diabetes]:
    # 1. No all-NaN columns in reference split
    assert (
        not bundle.X_ref.isnull().all(axis=0).any()
    ), f"{bundle.name}: found all-NaN column in reference split"

    # 2. Labels are binary {0, 1}
    for split_name, y in [
        ("ref", bundle.y_ref),
        ("pre", bundle.y_pre),
        ("post", bundle.y_post),
    ]:
        unique_vals = set(y.unique())
        assert unique_vals.issubset(
            {0, 1}
        ), f"{bundle.name}/{split_name}: labels are not binary, got {unique_vals}"

    # 3. All splits share the same columns
    assert (
        list(bundle.X_ref.columns)
        == list(bundle.X_pre.columns)
        == list(bundle.X_post.columns)
    ), f"{bundle.name}: column mismatch across splits"

    # 4. Feature lists partition all columns
    all_cols = set(bundle.X_ref.columns)
    declared = set(bundle.num_features) | set(bundle.cat_features)
    assert all_cols == declared, f"{bundle.name}: feature lists don't cover all columns"

    print(f"{bundle.name}: all sanity checks passed")

print("\nAll datasets OK!")

adult: all sanity checks passed
diabetes_readmission: all sanity checks passed

All datasets OK!


## 1.11 Compare with the `drift_framework` loader

The notebook re-implements the loader logic; the cell below verifies both produce identical output.


In [17]:
from drift_framework.data.loader import load_dataset as fw_load_dataset

fw_adult = fw_load_dataset("adult")

# Compare shapes
assert adult.X_ref.shape == fw_adult.X_ref.shape
assert adult.X_pre.shape == fw_adult.X_pre.shape
assert adult.X_post.shape == fw_adult.X_post.shape

print("Notebook loader matches drift_framework loader — shapes identical.")


Loading dataset: adult
Reference split  : X=(24420, 12),  class balance=0.240
Pre-drift split  : X=(8141, 12),  class balance=0.244
Post-drift split : X=(16281, 12), class balance=0.236
Numeric features  (6): ['Age', 'Race', 'Sex', 'Capital Gain', 'Capital Loss']...
Categorical features (6): ['Workclass', 'Education-Num', 'Marital Status', 'Occupation', 'Relationship']...
Notebook loader matches drift_framework loader — shapes identical.


c:\Users\jeffr\GitHub\pfc1\.venv\lib\site-packages\tableshift\datasets\adult.py:99: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Target'] = df['Target'].replace(
c:\Users\jeffr\GitHub\pfc1\.venv\lib\site-packages\sklearn\compose\_column_transformer.py:1590: UserWarning: Instantiating CategoricalDtype without any arguments.Pass a CategoricalDtype instance to silence this warning.
  df_row = df_row.select_dtypes(
c:\Users\jeffr\GitHub\pfc1\.venv\lib\site-packages\sklearn\compose\_column_transformer.py:1590: UserWarning: Instantiating CategoricalDtype without any arguments.Pass a CategoricalDtype instance to silence this warning.
  df_row = df_row.select_dtypes(
